# Module 06 — Convolutional Neural Networks
## 06-04 Transfer Learning & Fine-Tuning

**Objective:** Reuse the feature hierarchy learned on ImageNet via three transfer
strategies — linear probing, partial fine-tuning, and full fine-tuning — and
compare them against training from scratch on CIFAR-10.

**Prerequisites:** 06-03 (LeNet-5 & AlexNet), 06-04 (VGGNet), 06-05 (ResNet).


## Part 0 — Setup & Prerequisites

### Why transfer learning?

Training ResNet-18 on ImageNet takes days on multiple GPUs.  The resulting network
learns a rich **visual hierarchy**: early layers detect edges and colours; middle
layers detect textures and parts; late layers detect object-specific shapes.

Because natural images share the same low-level statistics regardless of the task,
these features **transfer** — even to datasets as different from ImageNet as CIFAR-10.

| Strategy | Frozen layers | Trainable layers | Speed | Peak accuracy |
|----------|--------------|-----------------|-------|---------------|
| **Linear probe** | All backbone | FC head only | Fastest | Good |
| **Feature extraction** | All backbone (end-to-end) | FC head only | Fast | Good |
| **Partial fine-tuning** | Early backbone | Late backbone + FC | Medium | Better |
| **Full fine-tuning** | None | All layers (low LR) | Slowest | Best |
| **From scratch** | — | All layers (normal LR) | Slow | Baseline |

In this notebook we:
1. Load **pretrained ResNet-18** from torchvision and adapt it for CIFAR-10.
2. Precompute backbone features and train a **linear probe** (no GPU, super fast).
3. Compare all four transfer strategies on a CIFAR-10 subset.
4. Visualise **feature separability** (PCA) for pretrained vs from-scratch models.
5. Analyse which layers need fine-tuning and which can stay frozen.


In [ ]:
import sys
import math
import time
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as tv_models

warnings.filterwarnings("ignore")

import random
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"NumPy       : {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA        : {torch.version.cuda}")
    print(f"GPU         : {torch.cuda.get_device_name(0)}")


In [ ]:

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
BATCH_SIZE    = 64      # smaller batch — 64x64 images are larger than 32x32
NUM_EPOCHS    = 5       # transfer learning converges fast
LEARNING_RATE = 1e-3    # full-lr for scratch; fine-tuning uses 1e-4
FT_LR         = 1e-4    # fine-tuning learning rate (lower to preserve pretrained weights)
NUM_CLASSES   = 10
DATA_ROOT     = "data/"
IMG_SIZE      = 64      # resize CIFAR-10 from 32x32; larger helps pretrained features
SUBSET_SIZE   = 15_000  # subset for end-to-end training (CPU speed)

# ImageNet normalisation — required for pretrained torchvision models
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

CIFAR_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

print(f"Image size    : {IMG_SIZE}x{IMG_SIZE}  (resized from 32x32)")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Subset size   : {SUBSET_SIZE:,}  (for end-to-end modes)")
print(f"LR (scratch)  : {LEARNING_RATE}")
print(f"LR (fine-tune): {FT_LR}")


### Dataset: CIFAR-10 with ImageNet Preprocessing

Pretrained torchvision models expect inputs normalised with **ImageNet statistics**
($\mu = (0.485, 0.456, 0.406)$, $\sigma = (0.229, 0.224, 0.225)$) and trained
on images roughly 224×224.  We resize CIFAR-10 to **64×64** — a practical
compromise between computation speed and feature quality on CPU.

> **Why ImageNet normalisation?**  The pretrained weights have learned to operate
> on inputs in this specific value range.  Using different normalisation would
> shift activations into an unexpected regime, degrading feature quality.


In [ ]:
# ── Transforms: ImageNet normalisation + resize ───────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=8),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train_aug = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True, transform=train_transform
)
full_train_val = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=False, transform=val_transform
)
test_set = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=True, transform=val_transform
)

generator     = torch.Generator().manual_seed(SEED)
all_indices   = torch.randperm(len(full_train_aug), generator=generator).tolist()
train_size    = int(0.9 * len(full_train_aug))
train_indices = all_indices[:train_size]
val_indices   = all_indices[train_size:]

# Full train set (for feature precomputation)
full_train_set = Subset(full_train_aug, train_indices)
val_set        = Subset(full_train_val, val_indices)

# Subset for end-to-end training modes (CPU speed)
subset_indices  = train_indices[:SUBSET_SIZE]
subset_train    = Subset(full_train_aug, subset_indices)

val_loader    = DataLoader(val_set,     batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=torch.cuda.is_available())
test_loader   = DataLoader(test_set,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=torch.cuda.is_available())
subset_loader = DataLoader(subset_train, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=torch.cuda.is_available())
full_loader   = DataLoader(full_train_set, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Full train : {len(full_train_set):,}")
print(f"Subset     : {len(subset_train):,}  (used for end-to-end fine-tuning)")
print(f"Val        : {len(val_set):,}")
print(f"Test       : {len(test_set):,}")
print(f"Image shape: (3, {IMG_SIZE}, {IMG_SIZE})")

# ── Sample grid ───────────────────────────────────────────────────────────────
raw_cifar = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False,
                                          download=False,
                                          transform=transforms.ToTensor())
class_imgs: dict[int, np.ndarray] = {}
for img_t, lbl in raw_cifar:
    if lbl not in class_imgs:
        class_imgs[lbl] = img_t.permute(1, 2, 0).numpy()
    if len(class_imgs) == NUM_CLASSES:
        break

fig_d, axes_d = plt.subplots(1, NUM_CLASSES, figsize=(15, 2.5))
for ci in range(NUM_CLASSES):
    axes_d[ci].imshow(class_imgs[ci])
    axes_d[ci].set_title(CIFAR_CLASSES[ci], fontsize=8)
    axes_d[ci].axis("off")
plt.suptitle("CIFAR-10 — One Sample per Class (raw 32x32)", fontsize=10,
             fontweight="bold")
plt.tight_layout()
plt.show()


## Part 1 — Pretrained ResNet-18: Anatomy & Adaptation

### ResNet-18 architecture recap

ResNet-18 (*see 06-05 for the residual block design*) has:

| Module | Description | Output size (224×224 in) |
|--------|-------------|--------------------------|
| `conv1` + `bn1` + `relu` + `maxpool` | Stem | 56×56×64 |
| `layer1` | 2 × BasicBlock(64) | 56×56×64 |
| `layer2` | 2 × BasicBlock(128, stride=2) | 28×28×128 |
| `layer3` | 2 × BasicBlock(256, stride=2) | 14×14×256 |
| `layer4` | 2 × BasicBlock(512, stride=2) | 7×7×512 |
| `avgpool` | AdaptiveAvgPool2d(1,1) | 1×1×512 |
| `fc` | Linear(512, 1000) | 1000 |

We replace the final `fc` with `Linear(512, 10)` for CIFAR-10.

### Feature extraction strategy

For **feature extraction**, we freeze the backbone (all layers except `fc`) and
only update the new classification head.  Freezing is done by setting
`requires_grad = False` on all backbone parameters.


In [ ]:
# ── Load pretrained ResNet-18 ─────────────────────────────────────────────────
def load_resnet18(pretrained: bool = True, num_classes: int = 10) -> nn.Module:
    '''Load ResNet-18 with optional ImageNet pretrained weights.

    Replaces the final FC layer with a new Linear(512, num_classes).
    The new FC is always trainable; pretrained backbone starts frozen by
    default (call unfreeze_backbone to enable full fine-tuning).

    Args:
        pretrained: Whether to load ImageNet pretrained weights.
        num_classes: Number of output classes for the new FC layer.

    Returns:
        Modified ResNet-18 nn.Module.
    '''
    if pretrained:
        model = tv_models.resnet18(weights="DEFAULT")
    else:
        model = tv_models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


resnet18_pretrained = load_resnet18(pretrained=True,  num_classes=NUM_CLASSES)
resnet18_scratch    = load_resnet18(pretrained=False, num_classes=NUM_CLASSES)

# Verify output shape
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    out = resnet18_pretrained(dummy)
print(f"ResNet-18 input:  {tuple(dummy.shape)}")
print(f"ResNet-18 output: {tuple(out.shape)}")

total_params  = sum(p.numel() for p in resnet18_pretrained.parameters())
fc_params     = sum(p.numel() for p in resnet18_pretrained.fc.parameters())
backbone_params = total_params - fc_params
print(f"\nTotal parameters  : {total_params:,}")
print(f"  Backbone params  : {backbone_params:,}  ({backbone_params/total_params*100:.1f}%)")
print(f"  FC head params   : {fc_params:,}  ({fc_params/total_params*100:.1f}%)")

# ── Named layer groups ─────────────────────────────────────────────────────────
layer_groups = ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4", "fc"]
print("\nLayer group parameter counts:")
print(f"  {'Group':<10s}  {'Params':>9s}  {'Trainable':>10s}")
print("-" * 36)
for grp in layer_groups:
    module = getattr(resnet18_pretrained, grp)
    n_all  = sum(p.numel() for p in module.parameters())
    n_tr   = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"  {grp:<10s}  {n_all:>9,}  {n_tr:>9,}")


### 1.2 Freezing Utilities

We build three helper functions to switch between transfer strategies:
- `freeze_backbone(model)` — only `fc` is trainable
- `unfreeze_stages(model, stages)` — unfreeze specific ResNet stages
- `unfreeze_all(model)` — full fine-tuning mode


In [ ]:
def freeze_backbone(model: nn.Module) -> None:
    '''Freeze all ResNet-18 layers except the final fc layer.

    Sets requires_grad=False for all backbone parameters,
    leaving only fc trainable (feature extraction mode).

    Args:
        model: ResNet-18 instance with a replaced fc head.
    '''
    for name, param in model.named_parameters():
        param.requires_grad = (name.startswith("fc."))


def unfreeze_stages(model: nn.Module, stages: list[str]) -> None:
    '''Unfreeze specific named top-level modules in the model.

    All other modules remain at their current requires_grad state.
    Typically called after freeze_backbone to enable partial fine-tuning.

    Args:
        model: ResNet-18 instance.
        stages: List of module names to unfreeze (e.g. ["layer3","layer4","fc"]).
    '''
    for name, param in model.named_parameters():
        top_module = name.split(".")[0]
        if top_module in stages:
            param.requires_grad = True


def unfreeze_all(model: nn.Module) -> None:
    '''Unfreeze all parameters for full fine-tuning.

    Args:
        model: Any PyTorch nn.Module.
    '''
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model: nn.Module) -> tuple[int, int]:
    '''Count trainable and total parameters.

    Args:
        model: PyTorch module.

    Returns:
        Tuple of (trainable_params, total_params).
    '''
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total


# ── Demonstrate each mode ─────────────────────────────────────────────────────
modes_demo = {
    "Feature extraction": lambda m: freeze_backbone(m),
    "Partial fine-tune":  lambda m: (freeze_backbone(m),
                                      unfreeze_stages(m, ["layer3","layer4","fc"])),
    "Full fine-tune":     lambda m: unfreeze_all(m),
    "From scratch":       lambda m: unfreeze_all(m),
}

print("Trainable parameters per transfer strategy:")
print(f"  {'Strategy':<22s}  {'Trainable':>11s}  {'Total':>11s}  {'%':>6s}")
print("-" * 58)
for strategy_name, setup_fn in modes_demo.items():
    m_demo = load_resnet18(pretrained=True, num_classes=NUM_CLASSES)
    result = setup_fn(m_demo)
    tr, tot = count_trainable(m_demo)
    print(f"  {strategy_name:<22s}  {tr:>11,}  {tot:>11,}  {tr/tot*100:>5.1f}%")

# ── Bar chart: trainable parameter fractions ─────────────────────────────────
fig_fp, ax_fp = plt.subplots(figsize=(8, 4))
strategies  = ["Feature\nextraction", "Partial\nfine-tune", "Full\nfine-tune", "From\nscratch"]
tr_fracs    = []
for setup_fn in modes_demo.values():
    m_d = load_resnet18(pretrained=True, num_classes=NUM_CLASSES)
    setup_fn(m_d)
    tr, tot = count_trainable(m_d)
    tr_fracs.append(tr / tot * 100)

bar_colors = ["#1f77b4", "#2ca02c", "#d62728", "#ff7f0e"]
bars_fp = ax_fp.bar(strategies, tr_fracs, color=bar_colors, edgecolor="k", lw=0.8)
for bar_f, frac in zip(bars_fp, tr_fracs):
    ax_fp.text(bar_f.get_x() + bar_f.get_width() / 2,
               bar_f.get_height() + 0.5,
               f"{frac:.1f}%", ha="center", va="bottom", fontsize=10)
ax_fp.set_ylabel("Trainable parameters (%)", fontsize=11)
ax_fp.set_title("Trainable Parameter Fraction per Transfer Strategy",
                fontsize=11, fontweight="bold")
ax_fp.set_ylim(0, 115)
ax_fp.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Part 2 — Linear Probe: Precomputed Feature Extraction

The fastest transfer strategy is **linear probing**:
1. Forward-pass the entire dataset through the **frozen** pretrained backbone once.
2. Save the resulting 512-dim feature vectors.
3. Train a simple `nn.Linear(512, 10)` on these cached features.

No GPU memory for the backbone is needed during training (only the linear layer).
This demonstrates whether ImageNet features are linearly separable for CIFAR-10.


In [ ]:
# ── Precompute backbone features ──────────────────────────────────────────────
def extract_features(
    backbone: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    '''Forward-pass all images through backbone and collect feature vectors.

    Args:
        backbone: Feature extractor (ResNet-18 without fc layer).
        dataloader: DataLoader over the dataset.
        device: Compute device.

    Returns:
        Tuple of (features, labels) as numpy arrays.
        features shape: (N, feature_dim).
        labels shape: (N,).
    '''
    backbone.eval()
    all_feats:  list[torch.Tensor] = []
    all_labels: list[torch.Tensor] = []

    with torch.no_grad():
        for images, labels in dataloader:
            feats = backbone(images.to(device))
            all_feats.append(feats.cpu())
            all_labels.append(labels)

    return (
        torch.cat(all_feats,  dim=0).numpy(),
        torch.cat(all_labels, dim=0).numpy(),
    )


# Build backbone: ResNet-18 without the fc layer
backbone_pretrained = load_resnet18(pretrained=True, num_classes=NUM_CLASSES)
backbone_pretrained.fc = nn.Identity()   # remove FC to get 512-dim vectors
backbone_pretrained.to(device)

# Build backbone from scratch for comparison
backbone_scratch = load_resnet18(pretrained=False, num_classes=NUM_CLASSES)
backbone_scratch.fc = nn.Identity()
backbone_scratch.to(device)

print("Extracting pretrained features (full training set)...")
t0 = time.time()
train_feats_pt, train_labels_pt = extract_features(backbone_pretrained, full_loader, device)
val_feats_pt,   val_labels_pt   = extract_features(backbone_pretrained, val_loader,  device)
test_feats_pt,  test_labels_pt  = extract_features(backbone_pretrained, test_loader, device)
print(f"  Done in {time.time()-t0:.1f}s  |  Feature dim: {train_feats_pt.shape[1]}")

print("Extracting scratch features (for PCA comparison)...")
t0 = time.time()
train_feats_sc, _ = extract_features(backbone_scratch, full_loader,  device)
val_feats_sc,   _ = extract_features(backbone_scratch, val_loader,   device)
print(f"  Done in {time.time()-t0:.1f}s")

print(f"\nFeature shapes: train={train_feats_pt.shape}  "
      f"val={val_feats_pt.shape}  test={test_feats_pt.shape}")

# ── PCA: pretrained vs scratch feature separability ───────────────────────────
fig_pca, axes_pca = plt.subplots(1, 2, figsize=(14, 6))

for ax, feats, title in [
    (axes_pca[0], val_feats_pt, "Pretrained ResNet-18 Features (PCA 2D)"),
    (axes_pca[1], val_feats_sc, "From-Scratch ResNet-18 Features (PCA 2D)"),
]:
    pca   = PCA(n_components=2, random_state=SEED)
    proj  = pca.fit_transform(feats)
    cmap  = plt.cm.get_cmap("tab10", NUM_CLASSES)
    for cls_idx in range(NUM_CLASSES):
        mask = val_labels_pt == cls_idx
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   s=6, alpha=0.5, color=cmap(cls_idx),
                   label=CIFAR_CLASSES[cls_idx])
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=9)
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.legend(fontsize=6, markerscale=3, loc="upper right",
              bbox_to_anchor=(1.3, 1.0))
    ax.grid(alpha=0.3)

plt.suptitle("ImageNet Pretrained vs Scratch Features on CIFAR-10 Validation Set",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Train linear probe on precomputed features ────────────────────────────────
PROBE_EPOCHS = 20   # linear probe is cheap — more epochs is fine

train_feat_ds = TensorDataset(torch.tensor(train_feats_pt, dtype=torch.float32),
                               torch.tensor(train_labels_pt, dtype=torch.long))
val_feat_ds   = TensorDataset(torch.tensor(val_feats_pt,   dtype=torch.float32),
                               torch.tensor(val_labels_pt,  dtype=torch.long))
test_feat_ds  = TensorDataset(torch.tensor(test_feats_pt,  dtype=torch.float32),
                               torch.tensor(test_labels_pt, dtype=torch.long))

probe_train_loader = DataLoader(train_feat_ds, batch_size=256, shuffle=True)
probe_val_loader   = DataLoader(val_feat_ds,   batch_size=256, shuffle=False)
probe_test_loader  = DataLoader(test_feat_ds,  batch_size=256, shuffle=False)

torch.manual_seed(SEED)
linear_probe = nn.Linear(train_feats_pt.shape[1], NUM_CLASSES)
probe_opt    = torch.optim.Adam(linear_probe.parameters(), lr=1e-3)
probe_crit   = nn.CrossEntropyLoss()

probe_val_accs:  list[float] = []
probe_val_losses: list[float] = []

print("Training linear probe on precomputed features...")
for epoch in range(PROBE_EPOCHS):
    linear_probe.train()
    for feats, labels in probe_train_loader:
        probe_opt.zero_grad()
        loss = probe_crit(linear_probe(feats), labels)
        loss.backward()
        probe_opt.step()

    linear_probe.eval()
    correct, total, run_loss = 0, 0, 0.0
    with torch.no_grad():
        for feats, labels in probe_val_loader:
            logits = linear_probe(feats)
            loss   = probe_crit(logits, labels)
            run_loss += loss.item() * labels.size(0)
            correct  += logits.argmax(1).eq(labels).sum().item()
            total    += labels.size(0)
    probe_val_accs.append(correct / total)
    probe_val_losses.append(run_loss / total)

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/{PROBE_EPOCHS} | "
              f"Val Acc: {probe_val_accs[-1]:.2%}")

# Test accuracy
linear_probe.eval()
correct, total = 0, 0
with torch.no_grad():
    for feats, labels in probe_test_loader:
        correct += linear_probe(feats).argmax(1).eq(labels).sum().item()
        total   += labels.size(0)
probe_test_acc = correct / total
print(f"\nLinear Probe Test Accuracy: {probe_test_acc:.2%}")
print(f"(Trained only a 512x10 = {512*10:,}-param linear layer in < 1 second)")


## Part 3 — End-to-End Transfer Strategies

### Training setup

We now compare four end-to-end training modes on the 15 K subset:

| Mode | Backbone init | Frozen? | LR |
|------|--------------|---------|-----|
| Feature extraction | Pretrained | All except fc | 1e-3 |
| Partial fine-tune | Pretrained | layer1, layer2 frozen | 1e-4 |
| Full fine-tune | Pretrained | Nothing frozen | 1e-4 |
| From scratch | Random | Nothing frozen | 1e-3 |

A lower learning rate for fine-tuning prevents overwriting the pretrained
knowledge with random gradient updates in the first few steps.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    '''Train for one epoch; return average loss and accuracy.

    Args:
        model: CNN to train.
        dataloader: Training DataLoader.
        criterion: Loss function.
        optimizer: Optimizer instance.
        device: Compute device.

    Returns:
        Tuple of (avg_loss, accuracy).
    '''
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += logits.argmax(1).eq(labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate model; return average loss and accuracy.

    Args:
        model: CNN to evaluate.
        dataloader: Validation or test DataLoader.
        criterion: Loss function.
        device: Compute device.

    Returns:
        Tuple of (avg_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)
            correct      += logits.argmax(1).eq(labels).sum().item()
            total        += labels.size(0)
    return running_loss / total, correct / total


In [ ]:
# ── Train all four transfer strategies ────────────────────────────────────────
criterion = nn.CrossEntropyLoss()

strategy_configs = [
    {
        "name":       "Feature extraction",
        "pretrained": True,
        "setup":      lambda m: freeze_backbone(m),
        "lr":         LEARNING_RATE,
    },
    {
        "name":       "Partial fine-tune",
        "pretrained": True,
        "setup":      lambda m: (freeze_backbone(m),
                                  unfreeze_stages(m, ["layer3", "layer4", "fc"])),
        "lr":         FT_LR,
    },
    {
        "name":       "Full fine-tune",
        "pretrained": True,
        "setup":      lambda m: unfreeze_all(m),
        "lr":         FT_LR,
    },
    {
        "name":       "From scratch",
        "pretrained": False,
        "setup":      lambda m: unfreeze_all(m),
        "lr":         LEARNING_RATE,
    },
]

all_histories: dict[str, dict] = {}

for cfg in strategy_configs:
    torch.manual_seed(SEED)
    model = load_resnet18(pretrained=cfg["pretrained"], num_classes=NUM_CLASSES)
    cfg["setup"](model)
    model.to(device)

    tr_params, tot_params = count_trainable(model)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr"],
    )

    train_losses: list[float] = []
    val_losses:   list[float] = []
    train_accs:   list[float] = []
    val_accs:     list[float] = []
    best_val_loss = float("inf")
    best_state    = None
    t_total       = 0.0

    print(f"\n{'='*62}")
    print(f"Strategy: {cfg['name']}  "
          f"(trainable={tr_params:,} / {tot_params:,}  "
          f"pretrained={'yes' if cfg['pretrained'] else 'no'})")
    print(f"{'='*62}")

    for epoch in range(NUM_EPOCHS):
        t0 = time.time()
        tl, ta = train_one_epoch(model, subset_loader, criterion, optimizer, device)
        vl, va = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - t0
        t_total += elapsed

        train_losses.append(tl); val_losses.append(vl)
        train_accs.append(ta);   val_accs.append(va)

        if vl < best_val_loss:
            best_val_loss = vl
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | "
              f"Val Acc: {va:.2%} | Time: {elapsed:.1f}s")

    model.load_state_dict(best_state)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"Test Accuracy: {test_acc:.2%}")

    all_histories[cfg["name"]] = {
        "model":        model,
        "train_losses": train_losses,
        "val_losses":   val_losses,
        "train_accs":   train_accs,
        "val_accs":     val_accs,
        "test_acc":     test_acc,
        "total_time":   t_total,
        "trainable":    tr_params,
        "pretrained":   cfg["pretrained"],
    }


In [ ]:
# ── Training curves: all strategies ──────────────────────────────────────────
strategy_colors = {
    "Feature extraction": "#1f77b4",
    "Partial fine-tune":  "#2ca02c",
    "Full fine-tune":     "#d62728",
    "From scratch":       "#ff7f0e",
}
epochs_x = range(1, NUM_EPOCHS + 1)

fig_tr, axes_tr = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in all_histories.items():
    col = strategy_colors[name]
    axes_tr[0].plot(epochs_x, hist["val_losses"],
                    color=col, lw=2, marker="o", markersize=4, label=name)
    axes_tr[0].plot(epochs_x, hist["train_losses"],
                    color=col, lw=1.2, ls="--", alpha=0.5)
    axes_tr[1].plot(epochs_x, [a * 100 for a in hist["val_accs"]],
                    color=col, lw=2, marker="o", markersize=4, label=name)
    axes_tr[1].plot(epochs_x, [a * 100 for a in hist["train_accs"]],
                    color=col, lw=1.2, ls="--", alpha=0.5)

for ax, title, ylabel in [
    (axes_tr[0], "Loss Curves (solid=val, dashed=train)", "Loss"),
    (axes_tr[1], "Val Accuracy (solid=val, dashed=train)", "Accuracy (%)"),
]:
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Transfer Learning Strategies — CIFAR-10 Training Curves",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary table
print(f"{'Strategy':<22s}  {'Ep1 Val':>8s}  {'Final Val':>10s}  "
      f"{'Test Acc':>9s}  {'Time':>7s}  {'Trainable':>11s}")
print("-" * 80)
# Linear probe line
print(f"  {'Linear probe':<20s}  {'—':>8s}  {probe_val_accs[-1]:>9.2%}  "
      f"{probe_test_acc:>8.2%}  {'<1s':>7s}  {'5,130':>11s}")
for name, hist in all_histories.items():
    print(f"  {name:<20s}  {hist['val_accs'][0]:>7.2%}  "
          f"{hist['val_accs'][-1]:>9.2%}  {hist['test_acc']:>8.2%}  "
          f"{hist['total_time']:>6.1f}s  {hist['trainable']:>11,}")


### 3.1 Discriminative Learning Rates

A key fine-tuning trick is **discriminative learning rates** — assigning a lower LR to early layers and a higher LR to later layers.  Early layers encode universal features (edges, textures) that require minimal adaptation; late layers encode task-specific semantics that need more updating.


In [ ]:
# ── Discriminative learning rates: different LRs per layer group ─────────────
# "Discriminative fine-tuning" (Howard & Ruder, ULMFiT 2018) assigns a
# lower LR to early layers and a higher LR to later layers.
# Early layers have universal features (no need to change much);
# late layers are task-specific and benefit from larger updates.

def build_discriminative_optimizer(
    model: nn.Module,
    base_lr: float = 1e-4,
    decay_factor: float = 0.3,
) -> torch.optim.Optimizer:
    '''Build an Adam optimizer with per-layer-group learning rates.

    Early layers get base_lr * decay_factor^(n_groups-1-i),
    i.e., the latest layers (layer4, fc) get the highest LR.

    Args:
        model: ResNet-18 instance.
        base_lr: Learning rate for the final layer group.
        decay_factor: Multiplicative decay from later to earlier groups.

    Returns:
        Adam optimizer with per-group learning rates.
    '''
    layer_groups = [
        ("conv1+bn1",  [model.conv1, model.bn1]),
        ("layer1",     [model.layer1]),
        ("layer2",     [model.layer2]),
        ("layer3",     [model.layer3]),
        ("layer4",     [model.layer4]),
        ("fc",         [model.fc]),
    ]
    n_groups = len(layer_groups)
    param_groups = []
    for group_idx, (group_name, modules) in enumerate(layer_groups):
        lr_i = base_lr * (decay_factor ** (n_groups - 1 - group_idx))
        params = []
        for m in modules:
            params.extend(p for p in m.parameters() if p.requires_grad)
        if params:
            param_groups.append({"params": params, "lr": lr_i,
                                  "name": group_name})
    return torch.optim.Adam(param_groups)


# ── Show LR schedule ──────────────────────────────────────────────────────────
print("Discriminative LR schedule (decay_factor=0.3, base_lr=1e-4):")
disc_model = load_resnet18(pretrained=True, num_classes=NUM_CLASSES)
unfreeze_all(disc_model)
disc_opt = build_discriminative_optimizer(disc_model, base_lr=1e-4, decay_factor=0.3)

group_names: list[str] = []
group_lrs:   list[float] = []
for grp in disc_opt.param_groups:
    print(f"  {grp['name']:<12s}  lr={grp['lr']:.2e}  "
          f"params={sum(p.numel() for p in grp['params']):,}")
    group_names.append(grp["name"])
    group_lrs.append(grp["lr"])

# ── Bar chart: LR per group ───────────────────────────────────────────────────
fig_dlr, ax_dlr = plt.subplots(figsize=(9, 4))
cols_dlr = plt.cm.viridis(np.linspace(0.15, 0.9, len(group_names)))
bars_dlr = ax_dlr.bar(group_names, group_lrs, color=cols_dlr, edgecolor="k", lw=0.8)
for bar_dlr, lr_dlr in zip(bars_dlr, group_lrs):
    ax_dlr.text(bar_dlr.get_x() + bar_dlr.get_width() / 2,
                bar_dlr.get_height() * 1.05,
                f"{lr_dlr:.1e}", ha="center", va="bottom", fontsize=9)
ax_dlr.set_ylabel("Learning rate", fontsize=11)
ax_dlr.set_title("Discriminative Fine-Tuning: Per-Layer-Group Learning Rate",
                  fontsize=11, fontweight="bold")
ax_dlr.set_yscale("log")
ax_dlr.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ── Train 3 epochs with discriminative LR and compare to uniform fine-tune ────
DISC_EPOCHS  = 3
disc_val_acc_per_epoch: list[float] = []
disc_crit = nn.CrossEntropyLoss()

disc_model.to(device)
disc_model.train()

print("\nTraining with discriminative LR (3 epochs, 15K subset)...")
for epoch in range(DISC_EPOCHS):
    disc_model.train()
    for images, labels in subset_loader:
        images, labels = images.to(device), labels.to(device)
        disc_opt.zero_grad()
        loss = disc_crit(disc_model(images), labels)
        loss.backward()
        disc_opt.step()

    disc_model.eval()
    correct_d, total_d = 0, 0
    with torch.no_grad():
        for imgs_d, lbls_d in val_loader:
            logits_d = disc_model(imgs_d.to(device))
            correct_d += logits_d.argmax(1).eq(lbls_d.to(device)).sum().item()
            total_d   += lbls_d.size(0)
    va_d = correct_d / total_d
    disc_val_acc_per_epoch.append(va_d)
    print(f"  Epoch {epoch+1}/{DISC_EPOCHS} | Val Acc: {va_d:.2%}")

# Compare disc LR epoch-3 vs full fine-tune epoch-3
full_ft_ep3 = all_histories["Full fine-tune"]["val_accs"][min(2, NUM_EPOCHS-1)]
print(f"\nEpoch-3 comparison:")
print(f"  Discriminative LR  : {disc_val_acc_per_epoch[-1]:.2%}")
print(f"  Uniform LR (1e-4)  : {full_ft_ep3:.2%}  (Full fine-tune from training above)")
print("\nDiscriminative LR often converges faster by preserving early-layer features.")


## Part 4 — Evaluation & Analysis

### 4.1 Convergence speed: which strategy wins at epoch 1?

Transfer learning's key advantage is **fast convergence** — pretrained features
make even epoch-1 accuracy much higher than from-scratch training.


In [ ]:
# ── Per-class accuracy comparison ────────────────────────────────────────────
all_preds_by_strategy: dict[str, list[int]] = {}
all_labels_ref: list[int] = []
labels_collected = False

for name, hist in all_histories.items():
    m = hist["model"]
    m.eval()
    preds_this: list[int] = []
    labels_this: list[int] = []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            logits = m(imgs.to(device))
            preds_this.extend(logits.argmax(1).cpu().tolist())
            labels_this.extend(lbls.tolist())
    all_preds_by_strategy[name] = preds_this
    if not labels_collected:
        all_labels_ref   = labels_this
        labels_collected = True

print("Per-class accuracy across strategies:")
header_strats = list(all_histories.keys())
print(f"  {'Class':<12s}", end="")
for s in header_strats:
    abbr = s.split()[0][:6]
    print(f"  {abbr:>7s}", end="")
print()
print("-" * (12 + len(header_strats) * 10 + 4))

for cls_idx in range(NUM_CLASSES):
    print(f"  {CIFAR_CLASSES[cls_idx]:<12s}", end="")
    for name in header_strats:
        preds = all_preds_by_strategy[name]
        mask  = [l == cls_idx for l in all_labels_ref]
        n_c   = sum(p == l for p, l, m in zip(preds, all_labels_ref, mask) if m)
        acc   = n_c / max(sum(mask), 1)
        print(f"  {acc:>6.2%} ", end="")
    print()

# ── Epoch-1 vs final accuracy bar chart ───────────────────────────────────────
fig_ep, axes_ep = plt.subplots(1, 2, figsize=(13, 4))
strategy_names = list(all_histories.keys())
ep1_accs   = [all_histories[n]["val_accs"][0]  * 100 for n in strategy_names]
final_accs = [all_histories[n]["test_acc"]      * 100 for n in strategy_names]
x_pos      = list(range(len(strategy_names)))
width      = 0.38
colors_ep  = [strategy_colors[n] for n in strategy_names]

for ax, accs, title_s in [
    (axes_ep[0], ep1_accs,   "Epoch-1 Val Accuracy (convergence speed)"),
    (axes_ep[1], final_accs, "Final Test Accuracy"),
]:
    bars_ep = ax.bar(x_pos, accs, color=colors_ep, edgecolor="k", lw=0.8)
    for bar_e, acc_e in zip(bars_ep, accs):
        ax.text(bar_e.get_x() + bar_e.get_width() / 2,
                bar_e.get_height() + 0.4,
                f"{acc_e:.1f}%", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(strategy_names, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy (%)", fontsize=10)
    ax.set_title(title_s, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 105); ax.grid(axis="y", alpha=0.3)

plt.suptitle("Transfer Learning: Convergence Speed vs Final Accuracy",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


### 4.1b Data Efficiency: The True Transfer Learning Advantage

Transfer learning's biggest impact is with **small labelled datasets**. We sweep training set sizes from 500 to 15K samples and compare feature extraction (frozen pretrained backbone) against from-scratch training.


In [ ]:
# ── Data efficiency: accuracy vs training set size ───────────────────────────
# Transfer learning shines most when labelled data is scarce.
# We compare feature extraction vs from-scratch at 4 data sizes:
# 500, 2000, 5000, and 15000 samples.

DATA_SIZES    = [500, 2_000, 5_000, 15_000]
DATA_EFF_EPOCHS = 3   # keep short for speed

data_eff_results: dict[str, dict[int, float]] = {
    "Feature extraction": {},
    "From scratch":       {},
}
crit_de = nn.CrossEntropyLoss()

for n_samples in DATA_SIZES:
    de_indices = train_indices[:n_samples]
    de_set     = Subset(full_train_aug, de_indices)
    de_loader  = DataLoader(de_set, batch_size=min(BATCH_SIZE, n_samples),
                             shuffle=True, num_workers=0)

    for strategy_de, pretrained_de, setup_de, lr_de in [
        ("Feature extraction", True,  freeze_backbone, LEARNING_RATE),
        ("From scratch",       False, unfreeze_all,    LEARNING_RATE),
    ]:
        torch.manual_seed(SEED)
        m_de = load_resnet18(pretrained=pretrained_de, num_classes=NUM_CLASSES)
        setup_de(m_de)
        m_de.to(device)
        opt_de = torch.optim.Adam(
            filter(lambda p: p.requires_grad, m_de.parameters()), lr=lr_de
        )

        for epoch in range(DATA_EFF_EPOCHS):
            m_de.train()
            for imgs_de, lbls_de in de_loader:
                imgs_de, lbls_de = imgs_de.to(device), lbls_de.to(device)
                opt_de.zero_grad()
                loss_de = crit_de(m_de(imgs_de), lbls_de)
                loss_de.backward()
                opt_de.step()

        _, va_de = evaluate(m_de, val_loader, crit_de, device)
        data_eff_results[strategy_de][n_samples] = va_de
        print(f"  n={n_samples:>6,}  {strategy_de:<22s}  val_acc={va_de:.2%}")

# ── Plot: data efficiency curves ──────────────────────────────────────────────
fig_de, ax_de = plt.subplots(figsize=(9, 5))
de_colors  = {"Feature extraction": "#1f77b4", "From scratch": "#ff7f0e"}
de_markers = {"Feature extraction": "o",       "From scratch": "^"}

for strategy_de, results_de in data_eff_results.items():
    sorted_items = sorted(results_de.items())
    xs = [item[0] for item in sorted_items]
    ys = [item[1] * 100 for item in sorted_items]
    ax_de.plot(xs, ys, marker=de_markers[strategy_de], lw=2,
               color=de_colors[strategy_de], markersize=8, label=strategy_de)
    for xi, yi in zip(xs, ys):
        ax_de.annotate(f"{yi:.1f}%", (xi, yi), textcoords="offset points",
                        xytext=(0, 8), ha="center", fontsize=8)

ax_de.set_xscale("log")
ax_de.set_xlabel("Training set size (log scale)", fontsize=11)
ax_de.set_ylabel("Val Accuracy (%)", fontsize=11)
ax_de.set_title("Data Efficiency: Feature Extraction vs From Scratch\n"
                 f"({DATA_EFF_EPOCHS} training epochs each)", fontsize=11,
                 fontweight="bold")
ax_de.legend(fontsize=10)
ax_de.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

# ── Print gap table ────────────────────────────────────────────────────────────
print("\nAccuracy gap (Feature extraction - From scratch) per data size:")
print(f"  {'n samples':>10s}  {'Feat extract':>13s}  {'From scratch':>13s}  {'Gap':>6s}")
print("-" * 50)
for n_s in DATA_SIZES:
    acc_ft = data_eff_results["Feature extraction"][n_s]
    acc_sc = data_eff_results["From scratch"][n_s]
    gap    = acc_ft - acc_sc
    print(f"  {n_s:>10,}  {acc_ft:>12.2%}  {acc_sc:>12.2%}  {gap:>+5.2%}")
print("\nTransfer learning gap is LARGEST at small data sizes.")
print("As data grows, from-scratch can eventually close the gap — but takes far more data.")


### 4.2 First-Layer Filter Visualisation: Pretrained vs Scratch

The first convolutional layer's learned filters reveal what low-level features
the model has detected.  Pretrained ResNet-18 filters resemble Gabor-like
orientation detectors; from-scratch filters are more noisy early in training.


In [ ]:
# ── First-layer filter visualisation: pretrained vs from-scratch ──────────────
def show_first_filters(
    model: nn.Module,
    title: str,
    n_show: int = 32,
) -> None:
    '''Plot the first conv layer filters as RGB images.

    Args:
        model: ResNet-18 model instance.
        title: Plot title string.
        n_show: Number of filters to display.
    '''
    w = model.conv1.weight.detach().cpu()   # (64, 3, 7, 7)
    n_cols = 8
    n_rows = math.ceil(n_show / n_cols)
    fig_fv, axes_fv = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
    axes_flat = axes_fv.ravel()

    for k in range(n_show):
        kern = w[k].permute(1, 2, 0).numpy()   # (H, W, 3)
        v_min, v_max = kern.min(), kern.max()
        kern_vis = (kern - v_min) / max(v_max - v_min, 1e-8)
        axes_flat[k].imshow(kern_vis.clip(0, 1))
        axes_flat[k].set_title(f"F{k}", fontsize=5)
        axes_flat[k].axis("off")

    for k in range(n_show, len(axes_flat)):
        axes_flat[k].axis("off")

    fig_fv.suptitle(title, fontsize=9, fontweight="bold")
    plt.tight_layout()
    plt.show()

    w_np = w.numpy()
    print(f"{title}: shape={w.shape}  mean={w_np.mean():.4f}  "
          f"std={w_np.std():.4f}  |w|_max={abs(w_np).max():.4f}")


pretrained_for_viz = load_resnet18(pretrained=True,  num_classes=NUM_CLASSES)
scratch_for_viz    = load_resnet18(pretrained=False, num_classes=NUM_CLASSES)

show_first_filters(pretrained_for_viz, "Pretrained ResNet-18 — First Conv Filters (32 of 64)")
show_first_filters(scratch_for_viz,    "From-Scratch ResNet-18 — First Conv Filters (32 of 64)")

# Similarity: how different are the filters?
w_pt = pretrained_for_viz.conv1.weight.detach().cpu().numpy().reshape(64, -1)
w_sc = scratch_for_viz.conv1.weight.detach().cpu().numpy().reshape(64, -1)
std_pt = w_pt.std(axis=1).mean()
std_sc = w_sc.std(axis=1).mean()
print(f"\nMean filter std (pretrained): {std_pt:.5f}")
print(f"Mean filter std (scratch)   : {std_sc:.5f}")
print(f"Pretrained filters are {std_pt/std_sc:.2f}x more spread out — Kaiming init vs ImageNet training")


### 4.2b Weight Drift: How Much Does Fine-Tuning Change Each Layer?

We measure the L2 distance between **pretrained weights** and **fine-tuned weights** per layer group.  This quantifies which layers adapt most during fine-tuning and validates the intuition that early layers should drift less.


In [ ]:
# ── Weight drift: how much does fine-tuning change each layer? ───────────────
# We measure the L2 distance between pretrained and fine-tuned weights for each
# ResNet-18 layer group.  Early layers should drift less (generic features);
# late layers drift more (adapting to CIFAR-10).

def compute_weight_drift(
    pretrained_model: nn.Module,
    finetuned_model: nn.Module,
    layer_names: list[str],
) -> dict[str, float]:
    '''Compute L2 weight drift per named top-level module.

    Args:
        pretrained_model: Model with original pretrained weights.
        finetuned_model: Model after fine-tuning.
        layer_names: List of top-level module names to compare.

    Returns:
        Dict mapping layer name to mean L2 weight drift.
    '''
    drift_map: dict[str, float] = {}
    for layer_name in layer_names:
        pt_params = list(getattr(pretrained_model, layer_name).parameters())
        ft_params = list(getattr(finetuned_model, layer_name).parameters())
        total_drift = 0.0
        n_params    = 0
        for pt_p, ft_p in zip(pt_params, ft_params):
            total_drift += (ft_p.detach() - pt_p.detach()).norm().item()
            n_params    += pt_p.numel()
        drift_map[layer_name] = total_drift / max(n_params, 1) * 1e4  # scaled
    return drift_map


# Reference: freshly loaded pretrained weights
pretrained_ref = load_resnet18(pretrained=True, num_classes=NUM_CLASSES)
layer_names    = ["conv1", "layer1", "layer2", "layer3", "layer4", "fc"]

# Drift for full fine-tune vs feature extraction
drift_full  = compute_weight_drift(
    pretrained_ref, all_histories["Full fine-tune"]["model"], layer_names
)
drift_feat  = compute_weight_drift(
    pretrained_ref, all_histories["Feature extraction"]["model"], layer_names
)
drift_partial = compute_weight_drift(
    pretrained_ref, all_histories["Partial fine-tune"]["model"], layer_names
)

print("Weight drift per layer (L2 norm / n_params x 1e4):")
print(f"  {'Layer':<10s}  {'Feat extract':>13s}  {'Partial FT':>11s}  {'Full FT':>8s}")
print("-" * 50)
for lname in layer_names:
    print(f"  {lname:<10s}  {drift_feat.get(lname, 0.0):>13.4f}  "
          f"{drift_partial.get(lname, 0.0):>11.4f}  "
          f"{drift_full.get(lname, 0.0):>8.4f}")

# ── Bar chart: drift per layer for full fine-tune ─────────────────────────────
fig_drift, ax_drift = plt.subplots(figsize=(10, 4))
x_d     = list(range(len(layer_names)))
width_d = 0.27
cols_d  = ["#1f77b4", "#2ca02c", "#d62728"]

for offset, (label_d, drift_dict) in enumerate([
    ("Feature extraction", drift_feat),
    ("Partial fine-tune",  drift_partial),
    ("Full fine-tune",     drift_full),
]):
    vals_d = [drift_dict.get(ln, 0.0) for ln in layer_names]
    bars_d = ax_drift.bar(
        [xi + offset * width_d for xi in x_d],
        vals_d, width_d, label=label_d, color=cols_d[offset], edgecolor="k", lw=0.5,
    )

ax_drift.set_xticks([xi + width_d for xi in x_d])
ax_drift.set_xticklabels(layer_names, fontsize=10)
ax_drift.set_ylabel("Weight drift (L2/n_params x 1e4)", fontsize=10)
ax_drift.set_title("Weight Drift vs Pretrained Init per Layer and Strategy",
                    fontsize=11, fontweight="bold")
ax_drift.legend(fontsize=9)
ax_drift.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  - Feature extraction: only 'fc' drifts (backbone frozen).")
print("  - Partial fine-tune: layer3/4/fc drift; early layers stay near pretrained.")
print("  - Full fine-tune: all layers drift, but early layers drift less than fc.")
print("  This confirms that pretrained early features remain largely intact even")
print("  after full fine-tuning — they generalise well across tasks.")


### 4.3 Final Comparison Table


In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
final_rows = [
    {
        "Strategy":     "Linear probe",
        "Pretrained":   "Yes",
        "Trainable":    f"{512*NUM_CLASSES:,}",
        "Ep-1 Val Acc": "—",
        "Test Acc":     f"{probe_test_acc:.4f}",
        "Train time":   "< 1s",
        "When to use":  "Quick evaluation of feature quality",
    }
]

for name, hist in all_histories.items():
    final_rows.append({
        "Strategy":     name,
        "Pretrained":   "Yes" if hist["pretrained"] else "No",
        "Trainable":    f"{hist['trainable']:,}",
        "Ep-1 Val Acc": f"{hist['val_accs'][0]:.4f}",
        "Test Acc":     f"{hist['test_acc']:.4f}",
        "Train time":   f"{hist['total_time']:.1f}s",
        "When to use":  {
            "Feature extraction": "Dataset very small; backbone trusted",
            "Partial fine-tune":  "Medium dataset; avoid overfitting early layers",
            "Full fine-tune":     "Larger dataset; maximum accuracy",
            "From scratch":       "No related pretrained model available",
        }.get(name, "—"),
    })

final_df = pd.DataFrame(final_rows)
print("Transfer learning strategy comparison:")
print(final_df.to_string(index=False))

# ── Accuracy progression: ep1 -> final for each strategy ─────────────────────
fig_fin, ax_fin = plt.subplots(figsize=(9, 5))
for name, hist in all_histories.items():
    col = strategy_colors[name]
    ax_fin.annotate(
        "", xy=(hist["test_acc"] * 100, 1), xytext=(hist["val_accs"][0] * 100, 0),
        arrowprops=dict(arrowstyle="->", color=col, lw=2)
    )
    ax_fin.scatter([hist["val_accs"][0] * 100], [0], color=col, s=80, zorder=5)
    ax_fin.scatter([hist["test_acc"] * 100],    [1], color=col, s=80,
                    marker="*", zorder=5)
    ax_fin.text(hist["val_accs"][0] * 100, -0.07, name.replace(" ", "\n"),
                ha="center", fontsize=8, color=col)
    ax_fin.text(hist["test_acc"] * 100,     1.07,
                f"{hist['test_acc']*100:.1f}%",
                ha="center", fontsize=8, color=col)

ax_fin.set_yticks([0, 1])
ax_fin.set_yticklabels(["Epoch 1 Val", "Final Test"], fontsize=10)
ax_fin.set_xlabel("Accuracy (%)", fontsize=11)
ax_fin.set_title("Accuracy Gain from Epoch 1 to Final Test\n"
                  "(arrows show improvement trajectory per strategy)",
                  fontsize=10, fontweight="bold")
ax_fin.set_ylim(-0.3, 1.35)
ax_fin.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **Pretrained features transfer well:** Even a frozen ImageNet backbone achieves
  strong CIFAR-10 accuracy with only a linear classifier on top.  The visual
  hierarchy (edges → textures → shapes) is universal across natural images.
- **Linear probing is the fastest sanity check:** Train only the FC head on
  precomputed features.  Achieves competitive accuracy in under 1 second and
  measures how **linearly separable** pretrained features are for the new task.
- **Convergence speed is the clearest transfer advantage:** Pretrained models
  achieve high accuracy after epoch 1; from-scratch models need many more epochs
  to learn the same representations.
- **Fine-tuning LR matters:** Full fine-tuning with a large LR can overwrite
  pretrained weights and perform *worse* than feature extraction.  Use a
  small LR (1e-4 vs 1e-3) to nudge rather than replace pretrained representations.
- **Partial fine-tuning is often the sweet spot:** Freeze the generic early layers
  (edges, textures), fine-tune the task-specific late layers (semantic features).
  This balances adaptation speed, parameter efficiency, and overfitting risk.

### What's Next

→ **06-07 (Data Augmentation)** pushes CIFAR-10 accuracy further by exploring
advanced augmentation strategies (Cutout, MixUp, CutMix) that act as strong
regularisers and effectively multiply dataset diversity at training time.
